In [ ]:
from sentence_transformers import SentenceTransformer
import torch

In [ ]:
import os
dir_path = os.getcwd()
print("The directory of this script is:", dir_path)
root_path = os.path.dirname(dir_path)
print("The root directory is:", root_path)

In [ ]:
import pandas as pd
medical_questions = pd.read_parquet(f"{root_path}\GraphRAG-Benchmark\Datasets\Questions\medical_questions.parquet")
medical_questions

In [ ]:
question_ids = medical_questions['id'].to_list()
question_ids

In [ ]:
import sys
sys.path.append(root_path)
from graphs.Node import Node
from answering.get_context import get_context
from answering.answer_prompt import answer_prompt

In [ ]:
# Load embedding model
device = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer("google/embeddinggemma-300m", device=device)

In [ ]:
#Load Data
import pickle
import json
import faiss
import pandas as pd
import numpy as np
#Graph - Node dict
with open(f"{root_path}/graphs/data/graphs/G6_entity_and_chunk_resolution_graph.pkl", "rb") as f:
    medical_graph = pickle.load(f)

#Embeddings
hnsw = faiss.read_index(f"{root_path}/graphs/data/embedding/medical_index.faiss")
with open(f"{root_path}/graphs/data/embedding/medical_ids.json", "r") as f:
    medical_embedding_ids = json.load(f)
num_vectors = hnsw.ntotal
dimension = hnsw.d
embeddings = np.zeros((num_vectors, dimension), dtype='float32')
for i in range(num_vectors):
    embeddings[i] = hnsw.reconstruct(i)

#Entities
with open(f"{root_path}/graphs/data/nodes/entity/medical_entities.pkl", "rb") as f:
    medical_entities = pickle.load(f)
medical_overview = pd.read_parquet(f"{root_path}/graphs/data/nodes/community/medical_communities_overview.parquet")

In [ ]:
#LLM api call
from google import genai
with open(f"{root_path}/API_KEY.txt", "r", encoding="utf-8") as f:
    API_KEY = f.read()

def call_gemini(text):
    client = genai.Client(api_key = API_KEY)
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=text
    )
    return response

In [ ]:
#Context setup
graph_context = {
    'graph': medical_graph,
    'entities': medical_entities,
    'overview': medical_overview
}

embedding_context = {
    'model': model,
    'index': hnsw,
    'ids': medical_embedding_ids,
}
ppr_context = {
    'k_ppr': None,
    'alpha': 0.5,
    't':5
}
query_context = {
    'question': None,
    'k_embedding': 8,
    'ppr': ppr_context
}


In [ ]:
try:
    medical_questions_answered = pd.read_parquet("data/medical_questions_answered.parquet")
except:
    medical_questions_answered = medical_questions.copy(deep=True)
    medical_questions_answered[["LLM_answer", "LLM_context", "LLM_tokens"]] = None


medical_questions_answered

In [ ]:
LLM_answer = (
    medical_questions_answered.loc[medical_questions_answered[["LLM_answer", "LLM_context", "LLM_tokens"]].notna().all(axis=1) &
        (medical_questions_answered[["LLM_answer", "LLM_context", "LLM_tokens"]] != "").all(axis=1)
        ].set_index("id")[["LLM_answer", "LLM_context", "LLM_tokens"]].to_dict(orient="index"))

LLM_answer


In [ ]:
from tqdm import tqdm
import time
sep = "\n\n" + "-"*100 + "\n\n"
MAX_RETRIES = 20
SLEEP_SEC = 30

def get_answers():
    for qid in tqdm(question_ids):
        if qid in LLM_answer:
            continue
        row = medical_questions.loc[medical_questions["id"] == qid].iloc[0]
        question = row["question"]
        query_context["question"] = question
        for attempt in range(1, MAX_RETRIES + 1):
            try:
                context = get_context(query_context,graph_context,embedding_context,API_KEY,debug=False)
                full_context = sep.join(context.values())
                prompt = answer_prompt(full_context, question)
                response = call_gemini(prompt)
                answer = response.text
                usage = response.usage_metadata
                #log
                LLM_answer[qid] = {
                    "LLM_answer": answer,
                    "LLM_context": full_context,
                    "LLM_tokens": usage.total_token_count,
                }
                break

            except Exception as e:
                code = e.error.code if hasattr(e, "error") else e
                print(f"[ERROR] qid={qid} failed after {attempt} retries: {code}")
                if attempt == MAX_RETRIES:
                    return
                time.sleep(SLEEP_SEC * attempt)  #backoff

get_answers()


In [ ]:
LLM_answer

In [ ]:
llm_df = (
    pd.DataFrame.from_dict(LLM_answer, orient="index")
      .reset_index()
      .rename(columns={"index": "id"})
)
llm_df

In [ ]:
print("min:", llm_df["LLM_tokens"].min())
print("max:", llm_df["LLM_tokens"].max())
print("avg:", llm_df["LLM_tokens"].mean())
print("sum:", llm_df["LLM_tokens"].sum())

In [ ]:
medical_questions_answered = medical_questions.merge(
    llm_df,
    on="id",
    how="left"
)
medical_questions_answered

In [ ]:
medical_questions_answered.to_parquet("data/medical_questions_answered.parquet")
medical_questions_answered_loaded = pd.read_parquet("data/medical_questions_answered.parquet")
medical_questions_answered_loaded